In [77]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, field_validator
from typing import Literal, List

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [78]:
# ─────────────────────────────────────────────────────────
# OUTPUT MODEL — what we want from the LLM
# ─────────────────────────────────────────────────────────

class EmailClassifierOutput(BaseModel):
    """
    Structured output from email classification.
    Every field has a description — this gets sent to the LLM!
    """
    category: Literal["Billing", "Technical", "General", "Spam","HUMAN_REVIEW_NEEDED"] 
    urgency: Literal["High", "Medium", "Low"] = Field(
        description="Urgency: High if financial impact >$100 or legal threat, Medium for moderate issues, Low for minor"
    )
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="Confidence in classification from 0.0 (unsure) to 1.0 (certain)"
    )
    summary: str = Field(
        description="One-sentence summary of the complaint including customer name if mentioned"
    )
    escalate: bool = Field(
        default=False,
        description="True only if complaint contains legal threats or fraud allegations"
    )
   




In [79]:
parser = PydanticOutputParser(pydantic_object=EmailClassifierOutput)

In [80]:
# first I am calling get_format_instructions to get the instructions

# String --> Rule + Schema definations(Output structure)
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "Structured output from email classification.\nEvery field has a description — this gets sent to the LLM!", "properties": {"category": {"enum": ["Billing", "Technical", "General", "Spam", "HUMAN_REVIEW_NEEDED"], "title": "Category", "type": "string"}, "urgency": {"description": "Urgency: High if financial impact >$100 or legal threat, Medium for moderate issues, Low for minor", "enum": ["High", "Medium", "Low"], "title": "Urgency", "type": "string"}, "confidence": {"description": "Confidence in classification from 0.0 (unsure) 

In [82]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """\
You are an expert email support classifier.
Analyze the email below and classify it according to the instructions.

{format_instructions}"""),
    ("human", "Subject: {subject}\n\nBody: {body}")
]).partial(
    # pre-fill format_instructions — only subject and body needed at runtime
    format_instructions=parser.get_format_instructions()
)

In [83]:
classifier_chain = prompt | llm | parser

In [84]:
test_emails = [
    {
        "subject": "Invoice #1042 — Wrong Amount",
        "body": "Hi, I'm John. I was charged $500 on invoice #1042 but we agreed on $250. Please fix this immediately."
    },
    {
        "subject": "Cannot Login",
        "body": "My account login has been broken since yesterday. I've tried resetting password but it still doesn't work."
    },
    {
        "subject": "Legal action warning",
        "body": "This is the third time you've charged me incorrectly. If not resolved today, I will contact my solicitor."
    },
]

In [85]:
for i, email in enumerate(test_emails):
    result = classifier_chain.invoke(email)
    print(f"Email {i+1}: {email['subject']}")
    print(f"Category: {result.category}")
    print(f"Urgency: {result.urgency}")
    print(f"Confidence: {result.confidence}")
    print(f"Summary: {result.summary}")
    print(f"Escalate: {result.escalate}")
    print()

Email 1: Invoice #1042 — Wrong Amount
Category: Billing
Urgency: High
Confidence: 0.9
Summary: John is disputing a charge of $500 on invoice #1042, claiming it should be $250.
Escalate: False

Email 2: Cannot Login
Category: Technical
Urgency: Medium
Confidence: 0.85
Summary: The customer is experiencing login issues with their account.
Escalate: False

Email 3: Legal action warning
Category: Billing
Urgency: High
Confidence: 0.9
Summary: Customer warns of legal action due to repeated incorrect charges.
Escalate: True



In [86]:
# category='Billing' urgency='High' confidence=0.9 summary='John reported being charged $500 on invoice #1042 instead of the agreed amount of $250.' escalate=False


In [87]:
bulk_emails = [
    {"subject": "Double charge",      "body": "Charged twice this month for the same subscription."},
    {"subject": "Feature request",    "body": "Would love dark mode in the dashboard."},
    {"subject": "App crashing",       "body": "The mobile app crashes every time I try to export data."},
    {"subject": "Congratulations!",   "body": "You've won a prize! Click here to claim it."},
]


In [88]:
results = classifier_chain.batch(bulk_emails)


In [89]:
results[0]

EmailClassifierOutput(category='Billing', urgency='Medium', confidence=0.9, summary='Customer reports being charged twice this month for the same subscription.', escalate=False)

In [90]:
bad_llm_output = "The email is about billing issues. Priority is high."

from langchain_core.exceptions import OutputParserException
try:
    parser.parse(bad_llm_output)
except OutputParserException as e:
    EmailClassifierOutput(
        category="HUMAN_REVIEW_NEEDED",
        urgency="High",
        confidence=0.0,
        summary="LLM output is not ok, need human review.",
        escalate=False
    )

In [91]:

def safe_classify(subject: str, body: str) -> dict:
    """Classify with graceful fallback on parse error."""
    try:
        # result = classifier_chain.invoke({"subject": subject, "body": body})
        result = "The email is about billing issues. Priority is high"
        parser.parse(result)
        return {"success": True, "result": result}
    except OutputParserException as e:
        return {
            "success": False,
            "error": str(e),
            "fallback": EmailClassifierOutput(
                category="General",
                urgency="Medium",
                confidence=0.0,
                summary="Classification failed — manual review needed",
                escalate=False
            )
        }


In [69]:
response = safe_classify("Payment failed", "I was charged wrong amount on invoice")

In [70]:
response

{'success': False,
 'error': 'Invalid json output: The email is about billing issues. Priority is high\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE ',
 'fallback': EmailClassifierOutput(category='General', urgency='Medium', confidence=0.0, summary='Classification failed — manual review needed', escalate=False)}

In [ ]:
response['fallback']

'General'

```
The complete flow:

  1. Define Pydantic model with Field(description=...)
         ↓
  2. PydanticOutputParser.get_format_instructions()
     → generates JSON schema instructions for LLM
         ↓
  3. Inject into prompt via {format_instructions}
         ↓
  4. LLM returns valid JSON matching the schema
         ↓
  5. Parser converts JSON → your Pydantic object
         ↓
  6. You get typed, validated Python object
     result.category, result.urgency — not string parsing!
```